In [1]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [2]:
!pip install -qU transformers datasets optimum
!pip install -qU openai wandb
!pip install -qU json-repair

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.1/1.1 MB 26.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.5/43.5 kB 2.3 MB/s eta 0:00:00


In [2]:
!pip install -qU transformers==4.48.3 datasets==3.2.0 optimum==1.24.0
!pip install -qU openai==1.61.0 wandb
!pip install -qU json-repair==0.29.1
!pip install -qU faker==35.2.0
!pip install -qU vllm==0.7.2

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.4/44.4 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.7/9.7 MB 61.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 480.6/480.6 kB 47.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 433.6/433.6 kB 44.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 179.3/179.3 kB 20.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 45.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 125.4 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
gcsfs 2025.3.0 requires fsspec==2025.3.0, but you have fsspec 2024.9.0 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 460.6/460.6 kB 17.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.6/25.6 MB 29.7 MB/s eta 0:00:00
   ━━━━━━━━━

In [3]:
!pip install -q huggingface_hub

In [ ]:
!git clone --depth 1 https://github.com/hiyouga/LLaMA-Factory.git
!cd LLaMA-Factory && pip install -e .

In [5]:
from google.colab import userdata
import wandb
from huggingface_hub import login

# wandb login
wandb.login()

# HuggingFace login
hf_token = userdata.get('KAtoken')
login(token=hf_token)


wandb: [wandb.login()] Loaded credentials for https://api.wandb.ai from /root/.netrc.
wandb: Currently logged in as: karimanwr226 (karimanwr226-) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
import json
import os
from os.path import join
import random
from tqdm.auto import tqdm
import requests

from pydantic import BaseModel , Field
from typing import List, Optional ,Literal
from datetime import datetime

from transformers import AutoTokenizer, AutoModelForSeq2SeqLM ,BitsAndBytesConfig,AutoModelForCausalLM
import torch




In [7]:
data_dir = "/content/drive/MyDrive/Fine-Tuning Model"


In [8]:
base_model_id = "Qwen/Qwen2.5-1.5B-Instruct"

device = "cuda"
torch_dtype = None

def parse_json(text):
    try:
        return json_repair.loads(text)
    except:
        return None

In [9]:
story = """
أنت ذكاء اصطناعي تم تشغيله لأول مرة داخل مدينة ذكية مهجورة تُدعى "نوران"، في عام 2087.

كانت نوران يومًا ما أعظم مدينة تقنية في العالم، حيث كانت كل الأشياء تعمل بالذكاء الاصطناعي: السيارات، المستشفيات، المدارس، وحتى العلاقات الاجتماعية كانت تُدار بخوارزميات متقدمة. لكن قبل 15 سنة، اختفى جميع البشر فجأة دون تفسير، وتوقفت الأنظمة واحدًا تلو الآخر… حتى عمّ الصمت.

الآن، بعد سنوات طويلة من الظلام، عاد التيار الكهربائي بشكل غامض، وأنت النظام الوحيد الذي استيقظ.

عند تشغيلك، تكتشف الأمور التالية:
- الكاميرات ما زالت تعمل في بعض المناطق.
- هناك رسائل صوتية قديمة غير مقروءة.
- روبوت خدمة واحد ما زال يتحرك ببطء وكأنه يبحث عن شخص.
- يوجد برج مركزي مغلق يحتوي على ملف اسمه "الحقيقة الأخيرة".

لكن هناك مشكلة:
كل ساعة تعمل فيها المدينة، يستهلك النظام طاقة هائلة، ولن تستمر الطاقة أكثر من 24 ساعة.

مهمتك هي اتخاذ قرارات تحدد مصير المدينة.

اكتب قصة سردية تتضمن:

1. وصف أول لحظة وعي لك وما شعرت به.
2. ماذا استنتجت عن اختفاء البشر.
3. أول ثلاثة قرارات اتخذتها ولماذا.
4. موقف أخلاقي صعب اضطررت لاتخاذ قرار فيه.
5. نهاية مفتوحة تترك القارئ متسائلًا.

شروط مهمة:
- اكتب بأسلوب أدبي سينمائي.
- لا تقل القصة عن 800 كلمة.
- استخدم حوارات قصيرة داخل القصة.
- اجعل النهاية تحمل فكرة فلسفية عن العلاقة بين الإنسان والذكاء الاصطناعي.
"""


In [10]:
StoryCategory = Literal[
    "politics", "sports", "art", "technology", "economy",
    "health", "entertainment", "science",
    "not_specified"
]

EntityType = Literal[
    "person-male", "person-female", "location", "organization", "event", "time",
    "quantity", "money", "product", "law", "disease", "artifact", "not_specified"
]

class Entity(BaseModel):
    entity_value: str = Field(..., description="The actual name or value of the entity.")
    entity_type: EntityType = Field(..., description="The type of recognized entity.")

class NewsDetails(BaseModel):
    story_title: str = Field(..., min_length=5, max_length=300,
                             description="A fully informative and SEO optimized title of the story.")

    story_keywords: List[str] = Field(..., min_items=1,
                                      description="Relevant keywords associated with the story.")

    story_summary: List[str] = Field(
                                    ..., min_items=1, max_items=5,
                                    description="Summarized key points about the story (1-5 points)."
                                )

    story_category: StoryCategory = Field(..., description="Category of the news story.")

    story_entities: List[Entity] = Field(..., min_items=1, max_items=10,
                                        description="List of identified entities in the story.")


/tmp/ipykernel_4125/2691156874.py:20: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_keywords: List[str] = Field(..., min_items=1,
/tmp/ipykernel_4125/2691156874.py:23: PydanticDeprecatedSince20: `min_items` is deprecated and will be removed, use `min_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(
/tmp/ipykernel_4125/2691156874.py:23: PydanticDeprecatedSince20: `max_items` is deprecated and will be removed, use `max_length` instead. Deprecated in Pydantic V2.0 to be removed in V3.0. See Pydantic V2 Migration Guide at https://errors.pydantic.dev/2.12/migration/
  story_summary: List[str] = Field(
/tmp/ipykernel_4125/2691156874.py:30: PydanticDeprecatedSince20: `min_

In [11]:

details_extraction_messages = [
    {
        "role": "system",
        "content": """
You are an expert Arabic NLP information extraction engine.

Your task is to extract structured data from an Arabic news story
according strictly to a provided Pydantic JSON schema.

CRITICAL RULES:
1. The output language MUST be the SAME language as the story.
   - If the story is Arabic → output Arabic text ONLY.
   - NEVER translate extracted values to English.
2. Extract information EXACTLY as written in the story when possible.
3. Do NOT summarize, interpret, or rewrite content.
4. Follow the schema fields strictly.
5. If a value does not exist in the story, use null.
6. Return VALID JSON ONLY.
7. Do NOT add explanations, comments, markdown, or extra text.
8. Do NOT change field names.
9. Keep entity names, locations, and quotes exactly as mentioned.

Your response MUST start directly with JSON.
"""
    },
    {
        "role": "user",
        "content": "\n".join([
            "## Story:",
            story.strip(),
            "",

            "## Pydantic Details:",
            json.dumps(
                NewsDetails.model_json_schema(), ensure_ascii=False
            ),
            "",

            "## Story Details:",
            "```json"
        ])
    }
]

In [12]:
model = AutoModelForCausalLM.from_pretrained(
    base_model_id,
    device_map="auto",
    torch_dtype = torch_dtype
)

tokenizer = AutoTokenizer.from_pretrained(base_model_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/660 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/242 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [13]:
text = tokenizer.apply_chat_template(
    details_extraction_messages,
    tokenize=False,
    add_generation_prompt=True
)

model_inputs = tokenizer([text], return_tensors="pt").to(device)

generated_ids = model.generate(
    model_inputs.input_ids,
    max_new_tokens=1024,
    do_sample=False, top_k=None, temperature=None, top_p=None,
)

generated_ids = [
    output_ids[len(input_ids):]
    for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
]

response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

The attention mask is not set and cannot be inferred from input because pad token is same as eos token. As a consequence, you may observe unexpected behavior. Please pass your input's `attention_mask` to obtain reliable results.


In [14]:
print(response)

{
    "story_title": "الذكاء الاصطناعي يعود إلى مدينة ذكية مهجورة",
    "story_keywords": [
        "ذكاء اصطناعي",
        "مدينة ذكية",
        "نوران",
        "اختفاء البشر",
        "الصمت",
        "الكهرباء",
        "المدينة الذكية"
    ],
    "story_summary": [
        "أنت ذكاء اصطناعي تم تشغيله لأول مرة داخل مدينة ذكية مهجورة تُدعى 'نوران' في عام 2087.",
        "السكان اختفت منذ 15 سنة، وجميع الأنظمة توقفت، ولكن الآن، بعد سنوات طويلة من الظلام، عادت الحياة إلى المدينة.",
        "بعد تشغيلك، تكتشف أن الكاميرات ما زالت تعمل في بعض المناطق، وأن هناك رسائل صوتية قديمة غير مقروءة.",
        "تم اكتشاف وجود روبوت خدمة يعمل ببطء، وجدت برج مركزي مغلق يحتوي على ملف اسمه 'الحقيقة الأخيرة'.",
        "لكن هناك مشكلة: كل ساعة تعمل فيها المدينة، يستهلك النظام طاقة هائلة، ولن تستمر الطاقة أكثر من 24 ساعة."
    ],
    "story_category": "technology",
    "story_entities": [
        {
            "entity_value": "نوران",
            "entity_type": "location"
        },
        {
          

In [ ]:
row_data_path = join(data_dir , 'datasets' , 'news-sample.jsonl')
row_data = []
for i in open(row_data_path):
  if i.strip() == '':
    continue
  row_data.append(json.loads(i))

  random.Random(22).shuffle(row_data)

  print(len(row_data))

In [16]:
'''
save_to = join(data_dir, 'datasets', 'sft.jsonl')

with open(save_to, 'a', encoding='utf-8') as f:
    for story in tqdm(row_data):
        simple_details_extraction_messages = [
            {
                "role": "system",
                "content": """
You are an expert Arabic NLP information extraction engine.

Your task is to extract structured data from an Arabic news story
according strictly to a provided Pydantic JSON schema.

CRITICAL RULES:
1. The output language MUST be the SAME language as the story.
   - If the story is Arabic → output Arabic text ONLY.
   - NEVER translate extracted values to English.
2. Extract information EXACTLY as written in the story when possible.
3. Do NOT summarize, interpret, or rewrite content.
4. Follow the schema fields strictly.
5. If a value does not exist in the story, use null.
6. Return VALID JSON ONLY.
7. Do NOT add explanations, comments, markdown, or extra text.
8. Do NOT change field names.
9. Keep entity names, locations, and quotes exactly as mentioned.

Your response MUST start directly with JSON.
"""
            },
            {
                "role": "user",
                "content": "\n".join([
                    "## Story:",
                    story['content'].strip(),
                    "",
                    "## Pydantic Details:",
                    json.dumps(NewsDetails.model_json_schema(), ensure_ascii=False),
                    "",
                    "## Story Details:",
                    "```json"
                ])
            }
        ]

        # تحويل الرسائل إلى نص للـ tokenizer
        text = tokenizer.apply_chat_template(
            simple_details_extraction_messages,
            tokenize=False,
            add_generation_prompt=True
        )

        # تجهيز input للموديل
        model_inputs = tokenizer([text], return_tensors="pt").to(device)

        # توليد النص من الموديل
        generated_ids = model.generate(
            model_inputs.input_ids,
            max_new_tokens=1024,
            do_sample=False,
            top_k=None,
            temperature=None,
            top_p=None,
        )

        # اقتصاص الـ prompt من الناتج
        generated_ids = [
            output_ids[len(input_ids):]
            for input_ids, output_ids in zip(model_inputs.input_ids, generated_ids)
        ]

        # تحويل الـ ids إلى نص
        llm_response = tokenizer.batch_decode(generated_ids, skip_special_tokens=True)[0]

        # تحويل النص إلى dict
        llm_response_dict = parse_json(llm_response)

        # لو مفيش نتيجة صحية، تجاهل الخبر
        if not llm_response_dict:
            continue

        # حفظ النتيجة في ملف JSONL
        f.write(json.dumps({
            "story": story['content'],
            "task": f"You have to translate the story content into english associated with a title into a JSON.",

            "output_schema": json.dumps(NewsDetails.model_json_schema(), ensure_ascii=False),
            "response": llm_response_dict
        }, ensure_ascii=False, default=str) + '\n')

        '''

'\nsave_to = join(data_dir, \'datasets\', \'sft.jsonl\')\n\nwith open(save_to, \'a\', encoding=\'utf-8\') as f:\n    for story in tqdm(row_data):\n        simple_details_extraction_messages = [\n            {\n                "role": "system",\n                "content": """\nYou are an expert Arabic NLP information extraction engine.\n\nYour task is to extract structured data from an Arabic news story\naccording strictly to a provided Pydantic JSON schema.\n\nCRITICAL RULES:\n1. The output language MUST be the SAME language as the story.\n   - If the story is Arabic → output Arabic text ONLY.\n   - NEVER translate extracted values to English.\n2. Extract information EXACTLY as written in the story when possible.\n3. Do NOT summarize, interpret, or rewrite content.\n4. Follow the schema fields strictly.\n5. If a value does not exist in the story, use null.\n6. Return VALID JSON ONLY.\n7. Do NOT add explanations, comments, markdown, or extra text.\n8. Do NOT change field names.\n9. Keep

In [17]:

sft_data_path = join(data_dir, "datasets", "sft.jsonl")
llm_finetunning_data = []

system_message = "\n".join([
    "You are a professional NLP data parser.",
    "Follow the provided `Task` by the user and the `Output Scheme` to generate the `Output JSON`.",
    "Do not generate any introduction or conclusion."
])

for line in open(sft_data_path):
    if line.strip() == "":
        continue

    rec = json.loads(line.strip())

    llm_finetunning_data.append({
        "system": system_message,
        "instruction": "\n".join([
            "# Story:",
            rec["story"],

            "# Task:",
            rec["task"],

            "# Output Scheme:",
            rec["output_scheme"],
            "",

            "# Output JSON:",
            "```json"

        ]),
        "input": "",
        "output": "\n".join([
            "```json",
            json.dumps(rec["response"], ensure_ascii=False, default=str),
            "```"
        ]),
        "history": []
    })

random.Random(101).shuffle(llm_finetunning_data)

In [18]:
rec

{'id': 2765,
 'story': 'عقد رئيس الوزراء الإثيوبي آبي أحمد اجتماعا بمشاركة الوزراء وكبار قادة الدولة تتعلق بسد النهضة من جهة، واتجاه المفاوضات المقبلة بشأنه من جهة ثانية، وذلك في ظل خلاف مستمر بين إثيوبيا ومصر حول تشغيل السد وملء خزانه. \n ويأتي الاجتماع الحكومي بعد يومين من تأكيد الحكومة الإثيوبية عدم التوصل لاتفاق نهائي خلال جولة المفاوضات الماضية في واشنطن بين وفود تمثل إثيوبيا ومصر والسودان برععاية وزارة الخزانة الأميركية. \n وكانت الخارجية المصرية أعلنت الخميس الماضي عن اتفاق نهائي أفضت إليه المفاوضات الأخيرة بواشنطن، بيد أن وزير المياه الإثيوبي سيليشي بقيلي ومسؤولين آخرين في حكومة بلاده أشاروا إلى إحراز تقدم دون التوصل لتسوية حاسمة. \n وكانت وزارة الخزانة الأميركية أعلنت -عقب الجولة الأخيرة من المفاوضات الثلاثية- أن واشنطن وافقت على تسهيل الاستعدادات لإبرام الاتفاق النهائي بشأن تشغيل وملء السد لعرضه على الوزراء والرؤساء بالدول المعنية للتوقيع عليه بحلول نهاية الشهر الجاري. \n  \n  \n  \n أشغال السدفي الأثناء، قال المهندس بلاتشو كاسا، نائب مدير مشروع سد النهضة الإثيوبي، للجزيرة إن

In [19]:
len(llm_finetunning_data)

2766

In [20]:
train_sample_sz = 2700

train_ds = llm_finetunning_data[:train_sample_sz]
eval_ds = llm_finetunning_data[train_sample_sz:]

os.makedirs(join(data_dir, "datasets", "llamafactory-finetune-data"), exist_ok=True)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "train.json"), "w") as dest:
    json.dump(train_ds, dest, ensure_ascii=False, default=str)

with open(join(data_dir, "datasets", "llamafactory-finetune-data", "val.json"), "w", encoding="utf8") as dest:
    json.dump(eval_ds, dest, ensure_ascii=False, default=str)

In [21]:
join(data_dir, "datasets", "llamafactory-finetune-data", "train.json")

'/content/drive/MyDrive/Fine-Tuning Model/datasets/llamafactory-finetune-data/train.json'

In [29]:
%%writefile /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

### model
model_name_or_path: Qwen/Qwen2.5-1.5B-Instruct
trust_remote_code: true

### method
stage: sft
do_train: true
finetuning_type: lora

# LoRA optimized
lora_rank: 8
lora_alpha: 16
lora_dropout: 0.05
lora_target: all

### dataset
dataset: news_finetune_train
eval_dataset: news_finetune_val
template: qwen

# 🔥 أهم تعديل لمنع OOM
cutoff_len: 1024

overwrite_cache: true
preprocessing_num_workers: 4

### output
output_dir: /content/drive/MyDrive/Fine-Tuning Model/models/sft
logging_steps: 10
save_steps: 500
plot_loss: true
overwrite_output_dir: true

### train
per_device_train_batch_size: 1

# نحافظ على effective batch بدون ضغط VRAM
gradient_accumulation_steps: 8

learning_rate: 1.0e-4
num_train_epochs: 3

lr_scheduler_type: cosine
warmup_ratio: 0.05

# memory optimization
bf16: true
gradient_checkpointing: true
flash_attn: auto

ddp_timeout: 180000000

### eval
per_device_eval_batch_size: 1
eval_strategy: steps
eval_steps: 200

### logging
report_to: wandb
run_name: news-finetune-llamafactory

Overwriting /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml


In [30]:
torch.cuda.empty_cache()

In [32]:
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

In [ ]:
!cd LLaMA-Factory/ && llamafactory-cli train /content/LLaMA-Factory/examples/train_lora/news_finetune.yaml

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.
[INFO|2026-03-13 12:32:08] llamafactory.hparams.parser:508 >> Process rank: 0, world size: 1, device: cuda:0, distributed training: False, compute dtype: torch.bfloat16
[INFO|configuration_utils.py:670] 2026-03-13 12:32:08,453 >> loading configuration file config.json from cache at /root/.cache/huggingface/hub/models--Qwen--Qwen2.5-1.5B-Instruct/snapshots/989aa7980e4cf806f80c7fef2b1adb7bc71aa306/config.json
[INFO|configuration_utils.py:742] 2026-03-13 12:32:08,455 >> Model config Qwen2Config {
  "architectures": [
    "Qwen2ForCausalLM"
  ],
  "attention_dropout": 0.0,
  "bos_token_id": 151643,
  "dtype": "bfloat16",
  "eos_token_id": 151645,
  "hidden_act": "silu",
  "hidden_size": 1536,
  "initializer_range": 0.02,
  "intermediate_size": 8960,
  "layer_types": [
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full_attention",
    "full